# Alle Artikel klassifizieren mit NewsBERT-germ-210m (Chunked)

Laedt das fine-tuned Modell von HuggingFace und klassifiziert Artikel
(train + test + raw) in konfigurierbaren Chunks.

**Sicherheitsmassnahmen:**
- HuggingFace Model-Cache wird beim Setup geloescht (frischer Download)
- `attn_implementation="eager"` erzwingt Standard-Attention statt SDPA (Fix fuer EuroBERT GPU-Bug)
- Label-Mapping wird gegen ALL_LABELS validiert
- Sanity-Check mit 5 bekannten Texten: Modell muss korrekte Klasse mit >40% Confidence vorhersagen
- Vorherige CSV wird automatisch verworfen wenn Mean Confidence <40% (Hinweis auf falsches Modell)

**Chunked-Modus:** `CHUNK_SIZE` legt fest, wie viele Artikel pro Durchlauf
klassifiziert werden. Bereits klassifizierte Artikel (aus vorherigem CSV)
werden uebersprungen. Notebook wiederholt ausfuehren, bis alle fertig sind.
`CHUNK_SIZE = 0` klassifiziert alle auf einmal.

**Output-Spalten:**
- Alle Original-Spalten aus dem HuggingFace-Datensatz
- `split` — Herkunft: train / test / raw
- `prediction` — Klasse mit hoechstem Score
- `prediction_score` — Confidence der Top-Prediction
- `text_length_char` — Textlaenge in Zeichen
- `text_length_token` — Textlaenge in Tokens (EuroBERT Tokenizer)
- `score_{Klasse}` — Softmax-Wahrscheinlichkeit fuer jede der 13 Klassen

**Voraussetzung:** GPU-Runtime (L4 empfohlen), `HF_TOKEN` in Colab Secrets.

In [ ]:
# === DEPENDENCIES (einmal ausfuehren, dann Restart) ===
# Colab hat transformers 5.0.0 vorinstalliert — diese Version hat einen Bug mit
# EuroBERTs custom modeling code. Upgrade auf >=5.2.0 ist zwingend noetig.
!pip install --upgrade "transformers[sentencepiece]>=5.2.0" datasets huggingface_hub \
    scikit-learn matplotlib seaborn tqdm pandas accelerate

import transformers
v = transformers.__version__
print(f"\ntransformers version: {v}")

if tuple(int(x) for x in v.split(".")[:2]) < (5, 2):
    print("Version zu alt — Runtime wird jetzt neugestartet...")
    print("Danach: diese Zelle UEBERSPRINGEN und direkt ab Zelle 2 (Setup) ausfuehren.")
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)
else:
    print("OK — weiter mit der naechsten Zelle.")

In [ ]:
# === SETUP ===
import os, sys, shutil

# Version pruefen (muss >=5.2.0 sein, installiert in Zelle 1)
import transformers
assert tuple(int(x) for x in transformers.__version__.split(".")[:2]) >= (5, 2), (
    f"transformers >= 5.2.0 benoetigt, aber {transformers.__version__} installiert!\n"
    f"Bitte Zelle 1 (Dependencies) ausfuehren und Runtime neustarten."
)
print(f"transformers: {transformers.__version__} (OK)")

REPO = "/content/news_articles_classification_thesis"
if not os.path.exists(REPO):
    !git clone https://github.com/ZorbeyOezcan/news_articles_classification_thesis.git {REPO}
else:
    !cd {REPO} && git pull -q

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

PIPELINE_DIR = f"{REPO}/Python/classification_pipeline"
if PIPELINE_DIR not in sys.path:
    sys.path.insert(0, PIPELINE_DIR)

import importlib
import pipeline_utils as pu
importlib.reload(pu)

from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))

# --- HuggingFace Model-Cache leeren ---
MODEL_REPO = "Zorryy/NewsBERT-germ-210m"
_hub_cache = os.path.expanduser(f"~/.cache/huggingface/hub/models--{MODEL_REPO.replace('/', '--')}")
_mod_cache = os.path.expanduser(f"~/.cache/huggingface/modules/transformers_modules/{MODEL_REPO.split('/')[0]}")

for _cache_path in [_hub_cache, _mod_cache]:
    if os.path.exists(_cache_path):
        shutil.rmtree(_cache_path)
        print(f"Cache geloescht: {_cache_path}")
    else:
        print(f"Kein Cache vorhanden: {_cache_path}")

print("\nSetup abgeschlossen.")

In [ ]:
# === KONFIGURATION ===
import torch
import numpy as np
import pandas as pd
from pathlib import Path

# MODEL_REPO bereits in Setup-Zelle definiert
DATASET_ID = pu.DATASET_ID  # "Zorryy/news_articles_2025_elections_germany"
BATCH_SIZE = 16  # Inference batch size
CHUNK_SIZE = 0  # Artikel pro Durchlauf (0 = alle auf einmal)

ALL_LABELS = [
    "Klima / Energie", "Zuwanderung", "Renten", "Soziales Gef\u00e4lle",
    "AfD/Rechte", "Arbeitslosigkeit", "Wirtschaftslage", "Politikverdruss",
    "Gesundheitswesen, Pflege", "Kosten/L\u00f6hne/Preise",
    "Ukraine/Krieg/Russland", "Bundeswehr/Verteidigung", "Andere",
]

# Score-Spaltennamen (Sonderzeichen entfernt)
def safe_col_name(label):
    return ("score_" + label
            .replace(" / ", "_").replace("/", "_").replace(", ", "_")
            .replace(" ", "_")
            .replace("\u00e4", "ae").replace("\u00f6", "oe").replace("\u00fc", "ue"))

SCORE_COLUMNS = [safe_col_name(l) for l in ALL_LABELS]

# Output
OUTPUT_DIR = Path(pu.REPORTS_DIR)
OUTPUT_CSV = OUTPUT_DIR / "all_articles_classified.csv"

# GPU check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    gpu_cap = torch.cuda.get_device_capability()
    print(f"GPU: {torch.cuda.get_device_name(0)} ({gpu_mem:.1f} GB, CC {gpu_cap[0]}.{gpu_cap[1]})")
    if gpu_mem >= 40:
        BATCH_SIZE = 64
    elif gpu_mem >= 20:
        BATCH_SIZE = 32
else:
    print("WARNUNG: Keine GPU — Inference wird sehr langsam!")

# KEIN AMP/autocast — EuroBERTs Custom Attention Code produziert NaN unter autocast.

print(f"Modell: {MODEL_REPO}")
print(f"Datensatz: {DATASET_ID}")
print(f"Batch Size: {BATCH_SIZE}")
print(f"Chunk Size: {CHUNK_SIZE if CHUNK_SIZE > 0 else 'ALLE'}")
print(f"Output: {OUTPUT_CSV}")

In [ ]:
# === DATENSATZ LADEN & CHUNK VORBEREITEN ===
from datasets import load_dataset

ds = load_dataset(DATASET_ID)
print(f"Splits: {list(ds.keys())}")
for split_name, split_ds in ds.items():
    print(f"  {split_name}: {len(split_ds):>7} Artikel")

# DataFrames mit split-Spalte erstellen
dfs = []
for split_name in ["train", "test", "raw"]:
    if split_name in ds:
        df = ds[split_name].to_pandas()
        df["split"] = split_name
        dfs.append(df)

all_df = pd.concat(dfs, ignore_index=True)
total_articles = len(all_df)
print(f"\nGesamt im Datensatz: {total_articles:,} Artikel")

# --- Vorherige Ergebnisse: Validierung oder Neustart ---
prev_df = None
if OUTPUT_CSV.exists():
    prev_df = pd.read_csv(OUTPUT_CSV)

    # Pruefen ob vorherige Ergebnisse plausibel sind (Confidence-Check)
    prev_mean_conf = prev_df["prediction_score"].mean()
    prev_max_conf = prev_df["prediction_score"].max()
    print(f"\nVorherige CSV gefunden: {len(prev_df):,} Zeilen")
    print(f"  Mean Confidence: {prev_mean_conf:.4f}, Max: {prev_max_conf:.4f}")

    if prev_mean_conf < 0.40:
        print(f"\n  WARNUNG: Mean Confidence ({prev_mean_conf:.4f}) ist verdaechtig niedrig!")
        print(f"  Die vorherige CSV wurde vermutlich mit einem fehlerhaften Modell erstellt.")
        print(f"  -> Vorherige Ergebnisse werden VERWORFEN. Starte komplett neu.\n")
        prev_df = None
        remaining_df = all_df.copy()
    else:
        already_done_ids = set(prev_df["id"].tolist())
        remaining_df = all_df[~all_df["id"].isin(already_done_ids)].reset_index(drop=True)
        print(f"  Vorherige Ergebnisse sehen plausibel aus (Confidence OK).")
        print(f"  Bereits klassifiziert: {len(already_done_ids):,}")
        print(f"  Noch offen:            {len(remaining_df):,}")
else:
    remaining_df = all_df.copy()
    print("Kein vorheriges Ergebnis gefunden — starte von vorne.")

# --- Chunk auswaehlen ---
if len(remaining_df) == 0:
    chunk_df = remaining_df
    print(f"\n{'='*50}")
    print(f"  ALLE {total_articles:,} ARTIKEL BEREITS KLASSIFIZIERT!")
    print(f"{'='*50}")
    print(f"Output: {OUTPUT_CSV}")
elif CHUNK_SIZE > 0 and len(remaining_df) > CHUNK_SIZE:
    chunk_df = remaining_df.iloc[:CHUNK_SIZE].copy()
    print(f"\nChunk: {len(chunk_df):,} von {len(remaining_df):,} verbleibenden Artikeln")
else:
    chunk_df = remaining_df.copy()
    if CHUNK_SIZE > 0:
        print(f"\nLetzter Chunk: {len(chunk_df):,} Artikel (weniger als CHUNK_SIZE={CHUNK_SIZE:,})")
    else:
        print(f"\nKein Chunking: alle {len(chunk_df):,} Artikel auf einmal")

if len(chunk_df) > 0:
    print(f"\nSplit-Verteilung im Chunk:")
    print(chunk_df["split"].value_counts().to_string())

In [ ]:
# === MODELL & TOKENIZER LADEN + CPU vs GPU DEBUG ===
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers.modeling_rope_utils import ROPE_INIT_FUNCTIONS

print(f"transformers version: {transformers.__version__}")
print(f"torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# EuroBERT RoPE Fix
def _default_rope_init(config, device=None, **kwargs):
    base = getattr(config, "rope_theta", 10000.0)
    partial_rotary_factor = getattr(config, "partial_rotary_factor", 1.0)
    head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)
    dim = int(head_dim * partial_rotary_factor)
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.int64).float().to(device) / dim))
    return inv_freq, 1.0

ROPE_INIT_FUNCTIONS["default"] = _default_rope_init

print(f"\nLade Modell: {MODEL_REPO}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_REPO, trust_remote_code=True, attn_implementation="eager"
)

# --- Label-Mapping pruefen ---
id2label = model.config.id2label
id2label_list = [id2label[i] for i in range(len(id2label))]
assert id2label_list == ALL_LABELS, f"Label-Mismatch!\n  {id2label_list}\n  {ALL_LABELS}"
print(f"Labels: {len(id2label)} (OK)")

# =====================================================================
#  DEBUG: CPU vs GPU Logit-Vergleich
# =====================================================================
TEST_TEXT = (
    "Die Bundesregierung plant neue Massnahmen zur Reduzierung der CO2-Emissionen "
    "und zum Ausbau erneuerbarer Energien im Rahmen des Klimaschutzgesetzes."
)
EXPECTED = "Klima / Energie"

inputs = tokenizer(TEST_TEXT, return_tensors="pt", max_length=2048, truncation=True)

# --- CPU ---
model_cpu = model.cpu()
model_cpu.eval()
inputs_cpu = {k: v.cpu() for k, v in inputs.items()}
with torch.no_grad():
    logits_cpu = model_cpu(**inputs_cpu).logits.float()
probs_cpu = torch.softmax(logits_cpu, dim=-1)
pred_cpu = id2label[probs_cpu.argmax(dim=-1).item()]
score_cpu = probs_cpu.max().item()

print(f"\n{'='*70}")
print(f"  CPU vs GPU DEBUG")
print(f"{'='*70}")
print(f"  Text: {TEST_TEXT[:80]}...")
print(f"  Erwartet: {EXPECTED}")
print(f"\n  CPU Prediction: {pred_cpu} ({score_cpu:.4f})")
print(f"  CPU Logits (erste 5): {logits_cpu[0, :5].tolist()}")

if torch.cuda.is_available():
    # --- GPU ---
    model_gpu = model_cpu.cuda()
    model_gpu.eval()
    inputs_gpu = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        logits_gpu = model_gpu(**inputs_gpu).logits.float()
    probs_gpu = torch.softmax(logits_gpu, dim=-1)
    pred_gpu = id2label[probs_gpu.argmax(dim=-1).item()]
    score_gpu = probs_gpu.max().item()

    print(f"\n  GPU Prediction: {pred_gpu} ({score_gpu:.4f})")
    print(f"  GPU Logits (erste 5): {logits_gpu[0, :5].cpu().tolist()}")

    # --- Differenz ---
    diff = (logits_cpu - logits_gpu.cpu()).abs()
    print(f"\n  Max Logit-Differenz (CPU vs GPU): {diff.max().item():.6f}")
    print(f"  Mean Logit-Differenz:             {diff.mean().item():.6f}")

    if diff.max().item() > 0.1:
        print(f"\n  !!! GROSSE DIFFERENZ — GPU produziert andere Logits als CPU !!!")
        print(f"  Alle CPU Logits:  {logits_cpu[0].tolist()}")
        print(f"  Alle GPU Logits:  {logits_gpu[0].cpu().tolist()}")

    # Modell fuer weitere Nutzung auf device belassen
    model = model_gpu
else:
    model = model_cpu

model.eval()
print(f"\n  Modell jetzt auf: {next(model.parameters()).device}")
print(f"{'='*70}")

# --- Sanity Check (alle 5 Texte) ---
SANITY_TESTS = [
    ("Die Bundesregierung plant neue Massnahmen zur Reduzierung der CO2-Emissionen "
     "und zum Ausbau erneuerbarer Energien im Rahmen des Klimaschutzgesetzes.",
     "Klima / Energie"),
    ("Die AfD hat bei den letzten Umfragen zugelegt und liegt nun bei 22 Prozent. "
     "Die Partei fordert eine haertere Migrationspolitik.",
     "AfD/Rechte"),
    ("Die Rentenreform wird heftig diskutiert. Viele Rentner fuerchten Kuerzungen "
     "ihrer Bezuege. Das Renteneintrittsalter soll angehoben werden.",
     "Renten"),
    ("Der Krieg in der Ukraine geht weiter. Russland hat erneut Raketenangriffe "
     "auf ukrainische Staedte gestartet. Die NATO beraet ueber weitere Hilfen.",
     "Ukraine/Krieg/Russland"),
    ("Die Bundeswehr braucht dringend neue Ausruestung und mehr Soldaten. "
     "Das Sondervermoegen fuer die Verteidigung wird aufgestockt.",
     "Bundeswehr/Verteidigung"),
]

MIN_CONFIDENCE = 0.40

print(f"\n{'='*70}")
print(f"  SANITY CHECK: {len(SANITY_TESTS)} Testbeispiele (auf {device})")
print(f"{'='*70}")

sanity_passed = 0
sanity_failed = 0

for text, expected_label in SANITY_TESTS:
    inputs = tokenizer(text, return_tensors="pt", max_length=2048, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits.float(), dim=-1)

    pred_id = probs.argmax(dim=-1).item()
    pred_label = id2label[pred_id]
    pred_score = probs[0, pred_id].item()

    ok = pred_label == expected_label and pred_score >= MIN_CONFIDENCE
    status = "PASS" if ok else "FAIL"
    if ok:
        sanity_passed += 1
    else:
        sanity_failed += 1

    print(f"\n  [{status}] Erwartet: {expected_label}")
    print(f"         Prediction: {pred_label} (score={pred_score:.4f})")

print(f"\n{'='*70}")
print(f"  Ergebnis: {sanity_passed}/{len(SANITY_TESTS)} bestanden, {sanity_failed} fehlgeschlagen")
print(f"{'='*70}")

if sanity_failed > 0:
    raise RuntimeError(
        f"\nSANITY CHECK FEHLGESCHLAGEN!\n"
        f"{sanity_failed}/{len(SANITY_TESTS)} Tests nicht bestanden.\n"
        f"Siehe CPU vs GPU Debug oben fuer Details."
    )

print("\nModell ist bereit fuer Batch-Inference.")

In [ ]:
# === TEXT-LAENGEN BERECHNEN ===
from tqdm import tqdm

# Char-Laenge
chunk_df["text_length_char"] = chunk_df["text"].fillna("").str.len()

# Token-Laenge (EuroBERT Tokenizer, in Batches fuer Speichereffizienz)
print("Berechne Token-Laengen...")
texts = chunk_df["text"].fillna("").tolist()
token_lengths = []
TOKEN_BATCH = 1000

for i in tqdm(range(0, len(texts), TOKEN_BATCH), desc="Tokenisiere"):
    batch = texts[i:i + TOKEN_BATCH]
    encoded = tokenizer(batch, truncation=False, add_special_tokens=False)
    token_lengths.extend(len(ids) for ids in encoded["input_ids"])

chunk_df["text_length_token"] = token_lengths

print(f"\nChar-Laenge:  min={chunk_df['text_length_char'].min()}, "
      f"max={chunk_df['text_length_char'].max()}, "
      f"median={chunk_df['text_length_char'].median():.0f}")
print(f"Token-Laenge: min={chunk_df['text_length_token'].min()}, "
      f"max={chunk_df['text_length_token'].max()}, "
      f"median={chunk_df['text_length_token'].median():.0f}")

In [ ]:
# === BATCH-INFERENCE (Dynamic Padding, kein AMP) ===
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding
from datasets import Dataset

MAX_LENGTH = 2048

texts = chunk_df["text"].fillna("").tolist()

print("Tokenisiere fuer Inference...")
hf_dataset = Dataset.from_dict({"text": texts})
hf_dataset = hf_dataset.map(
    lambda x: tokenizer(x["text"], max_length=MAX_LENGTH, truncation=True),
    batched=True, batch_size=1000, remove_columns=["text"],
)
hf_dataset.set_format("torch", columns=["input_ids", "attention_mask"])

# Dynamic Padding: jeder Batch wird nur auf die laengste Sequenz im Batch gepadded
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

dataloader = DataLoader(
    hf_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=data_collator, num_workers=4, pin_memory=True,
)

all_predictions = []
all_scores = []

print(f"Klassifiziere {len(texts):,} Artikel (Batch={BATCH_SIZE})...")

with torch.no_grad():
    for batch in tqdm(dataloader, desc="Inference"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits.float(), dim=-1).cpu().numpy()

        pred_ids = probs.argmax(axis=-1)
        pred_labels = [id2label[int(i)] for i in pred_ids]

        all_predictions.extend(pred_labels)
        all_scores.extend(probs.tolist())

print(f"\nInference abgeschlossen: {len(all_predictions):,} Predictions")

In [ ]:
# === ERGEBNISSE ZUSAMMENFUEGEN ===

# Prediction + Top-Score
chunk_df["prediction"] = all_predictions
scores_array = np.array(all_scores)
chunk_df["prediction_score"] = scores_array.max(axis=1)

# Score-Spalten fuer alle 13 Klassen
for i, col_name in enumerate(SCORE_COLUMNS):
    chunk_df[col_name] = scores_array[:, i]

# Spaltenreihenfolge festlegen
original_cols = ["id", "domain", "url", "date_time", "headline", "author",
                 "text", "text_length", "label", "split"]
new_cols = ["prediction", "prediction_score", "text_length_char", "text_length_token"]
final_cols = original_cols + new_cols + SCORE_COLUMNS
final_cols = [c for c in final_cols if c in chunk_df.columns]
chunk_df = chunk_df[final_cols]

print(f"Chunk-Ergebnisse: {chunk_df.shape[0]} Zeilen x {chunk_df.shape[1]} Spalten")

# --- Mit vorherigen Ergebnissen zusammenfuehren ---
if prev_df is not None and len(prev_df) > 0:
    combined_df = pd.concat([prev_df, chunk_df], ignore_index=True)
    print(f"Zusammengefuehrt: {len(prev_df):,} (vorher) + {len(chunk_df):,} (neu) = {len(combined_df):,}")
else:
    combined_df = chunk_df.copy()

print(f"\nVorschau (neue Predictions):")
display_cols = ["id", "headline", "split", "prediction", "prediction_score"]
display_cols = [c for c in display_cols if c in chunk_df.columns]
print(chunk_df[display_cols].head(3).to_string(index=False))

In [ ]:
# === CSV SPEICHERN ===
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
combined_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

file_size_mb = OUTPUT_CSV.stat().st_size / 1e6
print(f"CSV gespeichert: {OUTPUT_CSV}")
print(f"Groesse: {file_size_mb:.1f} MB")
print(f"Zeilen:  {len(combined_df):,} (von {total_articles:,} gesamt)")

remaining_after = total_articles - len(combined_df)
if remaining_after > 0:
    print(f"\nNoch offen: {remaining_after:,} Artikel")
    print(f"Notebook erneut ausfuehren fuer den naechsten Chunk!")
else:
    print(f"\nAlle {total_articles:,} Artikel klassifiziert!")

In [ ]:
# === SUMMARY STATS ===
from sklearn.metrics import accuracy_score, f1_score, classification_report

print("=" * 70)
print("  CLASSIFICATION SUMMARY")
print("=" * 70)

# --- 0. Fortschritt ---
print(f"\n0. Fortschritt:")
print(f"   Klassifiziert: {len(combined_df):>7,} / {total_articles:,}")
print(f"   Noch offen:    {total_articles - len(combined_df):>7,}")
print(f"   Dieser Chunk:  {len(chunk_df):>7,} Artikel")

# --- 1. Artikel pro Split ---
print(f"\n1. Artikel pro Split (klassifiziert):")
for split_name in ["train", "test", "raw"]:
    n = len(combined_df[combined_df["split"] == split_name])
    print(f"   {split_name:<6} {n:>7,}")
print(f"   {'TOTAL':<6} {len(combined_df):>7,}")

# --- 2. Prediction-Verteilung (alle klassifizierten Artikel) ---
print(f"\n2. Prediction-Verteilung (alle {len(combined_df):,} klassifizierten Artikel):")
pred_dist = combined_df["prediction"].value_counts().sort_values(ascending=False)
for label, count in pred_dist.items():
    pct = count / len(combined_df) * 100
    print(f"   {label:<30} {count:>7,} ({pct:>5.1f}%)")

# --- 3. Prediction-Verteilung (nur raw) ---
raw_df = combined_df[combined_df["split"] == "raw"]
if len(raw_df) > 0:
    print(f"\n3. Prediction-Verteilung (nur raw, {len(raw_df):,} Artikel):")
    raw_pred_dist = raw_df["prediction"].value_counts().sort_values(ascending=False)
    for label, count in raw_pred_dist.items():
        pct = count / len(raw_df) * 100
        print(f"   {label:<30} {count:>7,} ({pct:>5.1f}%)")

# --- 4. Uebereinstimmung Prediction vs. Label (train + test) ---
labeled_df = combined_df[combined_df["split"].isin(["train", "test"])].copy()
labeled_df = labeled_df[labeled_df["label"].notna() & (labeled_df["label"] != "")]

if len(labeled_df) > 0:
    acc = accuracy_score(labeled_df["label"], labeled_df["prediction"])
    f1_mac = f1_score(labeled_df["label"], labeled_df["prediction"],
                      average="macro", zero_division=0)
    f1_wt = f1_score(labeled_df["label"], labeled_df["prediction"],
                     average="weighted", zero_division=0)

    print(f"\n4. Uebereinstimmung Prediction vs. Label (train+test, n={len(labeled_df):,}):")
    print(f"   Accuracy:    {acc:.4f}")
    print(f"   F1 Macro:    {f1_mac:.4f}")
    print(f"   F1 Weighted: {f1_wt:.4f}")

    for split_name in ["train", "test"]:
        split_df = labeled_df[labeled_df["split"] == split_name]
        if len(split_df) > 0:
            s_acc = accuracy_score(split_df["label"], split_df["prediction"])
            s_f1 = f1_score(split_df["label"], split_df["prediction"],
                            average="macro", zero_division=0)
            print(f"   {split_name:<6} Accuracy={s_acc:.4f}, F1 Macro={s_f1:.4f} (n={len(split_df)})")

# --- 5. Confidence-Statistiken ---
print(f"\n5. Confidence-Statistiken (prediction_score):")
for split_name in ["train", "test", "raw"]:
    split_df = combined_df[combined_df["split"] == split_name]
    if len(split_df) > 0:
        scores = split_df["prediction_score"]
        print(f"   {split_name:<6} mean={scores.mean():.4f}, "
              f"median={scores.median():.4f}, "
              f"min={scores.min():.4f}, "
              f"<0.5={int((scores < 0.5).sum()):,} ({(scores < 0.5).mean()*100:.1f}%)")

# --- 6. Text-Laengen ---
print(f"\n6. Text-Laengen (klassifizierte Artikel):")
print(f"   Chars:  mean={combined_df['text_length_char'].mean():.0f}, "
      f"median={combined_df['text_length_char'].median():.0f}, "
      f"max={combined_df['text_length_char'].max():,}")
print(f"   Tokens: mean={combined_df['text_length_token'].mean():.0f}, "
      f"median={combined_df['text_length_token'].median():.0f}, "
      f"max={combined_df['text_length_token'].max():,}")

print(f"\n{'='*70}")
print(f"CSV: {OUTPUT_CSV}")
print(f"{'='*70}")

In [ ]:
# === PER-CLASS REPORT (train+test) ===
if len(labeled_df) > 0:
    print("Per-Class Classification Report (train+test vs. prediction):\n")
    print(classification_report(
        labeled_df["label"], labeled_df["prediction"],
        labels=ALL_LABELS, target_names=ALL_LABELS, zero_division=0,
    ))

In [ ]:
# === CLEANUP ===
import gc

del model, dataloader, hf_dataset, scores_array, all_scores, all_predictions
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

remaining_after = total_articles - len(combined_df)
if remaining_after > 0:
    print(f"Chunk fertig. {len(combined_df):,} / {total_articles:,} Artikel klassifiziert.")
    print(f"Noch {remaining_after:,} offen — Notebook erneut ausfuehren!")
else:
    print(f"Fertig! Alle {total_articles:,} Artikel klassifiziert.")
print(f"  {OUTPUT_CSV}")